In [16]:
from __future__ import annotations

import json
import zipfile
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score, 
    recall_score
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

In [17]:
DATA_PATH = './data/datasets/sinistros/datatran2025.csv'

In [18]:
# Funções auxiliares para manipular o dataset

def load_dataset(path: Path, sep: str = None, encoding: str = "latin1") -> pd.DataFrame:
    df = pd.read_csv(path, sep=sep, engine="python", encoding=encoding)
    return df


def preprocess_dataset(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    for column in [
        "mortos",
        "feridos_leves",
        "feridos_graves",
        "ilesos",
        "feridos",
        "ignorados",
    ]:
        df[f"teve_{column}"] = df[column] > 0

    return df

# Funções auxiliares para trabalhar com os dados de treino e teste

# Adicionar o controle de sparse para evitar erros do Iterative e do Knn imputers, relacionados ao recebimento de 
# dados esparsos, que quebrava a execução dos pipelines.
def make_one_hot_encoder(sparse: bool = True) -> OneHotEncoder:
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=sparse)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=sparse)

def split_X_y(df: pd.DataFrame, target: str, extra: list[str]) -> tuple[pd.DataFrame, pd.Series]:
    if target not in df.columns:
        raise ValueError(f"Target '{target}' nao esta no dataset.")
    return df.drop(columns=[target] + extra), df[target]

# Tokens que representam valores ausentes
NUMERIC_MISSING_VALUES = [-200, -200.0]

STRING_MISSING_VALUES = [
    "?", " ?", "? ", "NA", "N/A", "na", "n/a", "NaN", "nan", "", " ",
    "unknown", "Unknown", "-200",
]

def normalize_missing_values(df: pd.DataFrame) -> pd.DataFrame:
    """Substitui tokens de missing por NaN, coluna a coluna, respeitando o dtype.
    
    - Colunas numericas: substitui -200 e -200.0
    - Colunas de texto: faz strip e substitui tokens como '?', 'unknown', etc.
    Essa abordagem evita erros do pandas 2.x ao misturar int e str no replace().
    -- TODO: verificar se tem outros simbolos estranhos nos dataset da atividade (pedir par aos alunos checarem isso)
    """
    df = df.copy()
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            df[col] = df[col].replace(NUMERIC_MISSING_VALUES, np.nan)
        else:
            # Normaliza os tokens de ausente para np.nan antes do imputador,
            # porque pd.NA nesta base gera erro no pipeline com scikit-learn/pandas.
            # Exemplo do SimpleImputer: "boolean value of NA is ambiguous"
            df[col] = df[col].astype(object)
            df[col] = df[col].apply(lambda v: v.strip() if isinstance(v, str) else v)
            df[col] = df[col].replace(STRING_MISSING_VALUES, np.nan)
    return df



# Pré-Processamento

In [19]:
df = preprocess_dataset(load_dataset(DATA_PATH))

df

,id,data_inversa,dia_semana,horario,uf,br,km,municipio,causa_acidente,tipo_acidente,classificacao_acidente,fase_dia,sentido_via,condicao_metereologica,tipo_pista,tracado_via,uso_solo,pessoas,mortos,feridos_leves,feridos_graves,ilesos,ignorados,feridos,veiculos,latitude,longitude,regional,delegacia,uop,teve_mortos,teve_feridos_leves,teve_feridos_graves,teve_ilesos,teve_feridos,teve_ignorados
0,652493,2025-01-01,quarta-feira,06:20:00,SP,116,225,GUARULHOS,Reação tardia ou ineficiente do condutor,Tombamento,Com Vítimas Feridas,Pleno dia,Decrescente,Céu Claro,Múltipla,Reta;Declive,Sim,2,0,1,0,0,1,1,2,"-23,48586772","-46,54075317",SPRF-SP,DEL01-SP,UOP01-DEL01-SP,False,True,False,False,True,True
1,652519,2025-01-01,quarta-feira,07:50:00,CE,116,"546,2",PENAFORTE,Pista esburacada,Colisão frontal,NaN,Pleno dia,Crescente,Céu Claro,Simples,Reta,Não,6,1,1,0,1,4,1,6,"-7,812288","-39,08333306",SPRF-CE,DEL05-CE,UOP03-DEL05-CE,True,True,False,True,True,True
2,652522,2025-01-01,quarta-feira,08:45:00,PR,369,"88,2",CORNELIO PROCOPIO,Reação tardia ou ineficiente do condutor,Colisão traseira,Com Vítimas Feridas,Pleno dia,Crescente,Sol,Dupla,Reta;Aclive,Sim,5,0,3,0,2,0,3,2,"-23,182565","-50,637228",SPRF-PR,DEL07-PR,UOP05-DEL07-PR,False,True,False,True,True,False
3,652544,2025-01-01,quarta-feira,11:00:00,PR,116,74,CAMPINA GRANDE DO SUL,Reação tardia ou ineficiente do condutor,Saída de leito carroçável,Com Vítimas Feridas,Pleno dia,Crescente,Céu Claro,Dupla,Reta,Não,5,0,1,0,4,0,1,2,"-25,36517687","-49,04223028",SPRF-PR,DEL01-PR,UOP02-DEL01-PR,False,True,False,True,True,False
4,652549,2025-01-01,quarta-feira,09:30:00,MG,251,471,FRANCISCO SA,Velocidade Incompatível,Colisão frontal,Com Vítimas Feridas,Pleno dia,Decrescente,Chuva,Simples,Curva;Declive,Não,5,0,1,1,1,2,2,4,"-16,46801304","-43,43121303",SPRF-MG,DEL12-MG,UOP01-DEL12-MG,False,True,True,True,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
72524,757462,2025-12-20,sábado,13:10:00,MG,381,"279,5",JAGUARACU,Acessar a via sem observar a presença dos outr...,Colisão transversal,Com Vítimas Feridas,Pleno dia,Decrescente,Céu Claro,Simples,Interseção de Vias,Não,3,0,2,0,1,0,2,2,"-19,607","-42,7611",SPRF-MG,DEL03-MG,UOP02-DEL03-MG,False,True,False,True,True,False
72525,757492,2025-09-27,sábado,13:50:00,SC,282,381,HERVAL DOESTE,Conversão proibida,Colisão transversal,Com Vítimas Feridas,Pleno dia,Decrescente,Céu Claro,Simples,Reta;Declive,Não,4,0,1,1,0,2,2,3,"-27,2096","-51,50077",SPRF-SC,DEL07-SC,UOP05-DEL07-SC,False,True,True,False,True,True
72526,757593,2025-12-14,domingo,13:50:00,PE,104,62,CARUARU,Ausência de reação do condutor,Colisão transversal,Com Vítimas Feridas,Pleno dia,Crescente,Céu Claro,Dupla,Reta,Sim,3,0,1,0,2,0,1,2,"-8,25368","-35,97546",SPRF-PE,DEL02-PE,UOP01-DEL02-PE,False,True,False,True,True,False
72527,758175,2025-12-15,segunda-feira,15:50:00,SC,101,130,CAMBORIU,Condutor deixou de manter distância do veículo...,Colisão traseira,Sem Vítimas,Pleno dia,Crescente,Sol,Dupla,Reta,Sim,2,0,0,0,2,0,0,2,"-26,99305955","-48,6609548",SPRF-SC,DEL04-SC,UOP03-DEL04-SC,False,False,False,True,False,False


In [20]:
boolean_cols = df.select_dtypes(include="bool").columns.tolist()

bool_counts = pd.DataFrame(
    {
        "True": df[boolean_cols].sum(),
        "False": (~df[boolean_cols]).sum(),
    }
).astype(int)

display(bool_counts)

categorical_columns = [
    "tipo_acidente",
    "classificacao_acidente",
    "fase_dia",
    "causa_acidente",
    "condicao_metereologica",
    "tipo_pista",
    "tracado_via",
    "regional"
]

unique_values = {
    column: sorted(df[column].dropna().unique().tolist())
    for column in categorical_columns
    if column in df.columns
}

unique_values_df = pd.DataFrame(
    {
        "coluna": list(unique_values.keys()),
        "valores_unicos": list(unique_values.values()),
    }
 )

display(unique_values_df)

,True,False
teve_mortos,5210,67319
teve_feridos_leves,45966,26563
teve_feridos_graves,16354,56175
teve_ilesos,45198,27331
teve_feridos,58042,14487
teve_ignorados,19857,52672


,coluna,valores_unicos
0,tipo_acidente,"[Atropelamento de Animal, Atropelamento de Ped..."
1,classificacao_acidente,"[Com Vítimas Fatais, Com Vítimas Feridas, Sem ..."
2,fase_dia,"[Amanhecer, Anoitecer, Plena Noite, Pleno dia]"
3,causa_acidente,[Acessar a via sem observar a presença dos out...
4,condicao_metereologica,"[Chuva, Céu Claro, Garoa/Chuvisco, Ignorado, N..."
5,tipo_pista,"[Dupla, Múltipla, Simples]"
6,tracado_via,"[Aclive, Aclive;Curva, Aclive;Curva;Em Obras, ..."
7,regional,"[SPRF-AC, SPRF-AL, SPRF-AM, SPRF-AP, SPRF-BA, ..."


# Teste baseline com dropna

Para testar o desempenho do dataset sem nenhum pré-processamento mais intensivo e integração com fontes externas, será criado um baseline com dropna.

In [21]:

def build_preprocessor_baseline(X: pd.DataFrame, scale_numeric: bool) -> ColumnTransformer:
    numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    categorical_cols = [c for c in X.columns if c not in numeric_cols]
    return ColumnTransformer(
        transformers=[
            ("num", StandardScaler() if scale_numeric else "passthrough", numeric_cols),
            ("cat", make_one_hot_encoder(), categorical_cols),
        ],
        remainder="drop",
    )


def run_classification_baseline(X: pd.DataFrame, y: pd.Series, test_size: float, random_state: int) -> dict:
    class_counts = y.value_counts(dropna=False)
    stratify = y if class_counts.min() >= 2 else None
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state, stratify=stratify
    )

    models = {
        "knn": Pipeline([
            ("preprocess", build_preprocessor_baseline(X_train, scale_numeric=True)),
            ("model", KNeighborsClassifier(n_neighbors=5, n_jobs=-1)),
        ]),
        "random_forest": Pipeline([
            ("preprocess", build_preprocessor_baseline(X_train, scale_numeric=False)),
            ("model", RandomForestClassifier(
                n_estimators=200,
                random_state=random_state,
                n_jobs=-1,
                class_weight="balanced_subsample",
            )),
        ]),
    }

    results = {}
    for name, model in models.items():
        model.fit(X_train, y_train)
        pred = model.predict(X_test)

        metrics = {
            "accuracy": float(accuracy_score(y_test, pred)),
            "macro_f1": float(f1_score(y_test, pred, average="macro")),
            "weighted_f1": float(f1_score(y_test, pred, average="weighted")),
            # Como o weighted está sendo usado para escolher o melhor modelo, vou coletar o precision e o recall para ter uma noção melhor do porquê o f1 deu o valor que deu
            "weighted_precision": float(precision_score(y_test, pred, average="weighted")),
            "weighted_recall": float(recall_score(y_test, pred, average="weighted")),
        }
        results[name] = metrics

        print(f"\n[model] {name}")
        print(metrics)
        print("Matriz de confusao:")
        print(confusion_matrix(y_test, pred))
        print(classification_report(y_test, pred))

    return results


def run_one_dataset_with_drop(key: str, target, out_dir: str = "data", test_size: float = 0.2, random_state: int = 42) -> dict:
    data_dir = Path(out_dir)
    df = preprocess_dataset(load_dataset(data_dir / f"{key}.csv"))

    print("\n" + "=" * 80)
    print(f"Dataset: {key} | Target: {target}")
    print("=" * 80)
    print("Shape original:", df.shape)

    df = normalize_missing_values(df)
    
    rows_before = len(df)
    df_clean = df.dropna(axis=0, how="any").copy()
    rows_after = len(df_clean)

    rows_dropped = rows_before - rows_after

    print("Linhas removidas no dropna:", rows_dropped, f"({(rows_dropped/rows_before):.2%})")

    if rows_after == 0:
        raise RuntimeError("Dataset vazio apos dropna.")

    X, y = split_X_y(df_clean, target, ['mortos', 'delegacia', 'uop'])

    print("Distribuicao da target:")
    print(y.value_counts())
    results = run_classification_baseline(X, y, test_size=test_size, random_state=random_state)

    result_dir = data_dir / "results" / "baseline"
    result_dir.mkdir(parents=True, exist_ok=True)
    out_path = result_dir / f"{key}_baseline_results.json"
    out_path.write_text(
        json.dumps(
            {
                "key": key,
                "target": target,
                "variant": "default",
                "shape_original": list(df.shape),
                "shape_after_prepro": list(df_clean.shape),
                "test_size": test_size,
                "random_state": random_state,
                "results": results,
            },
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )
    print("Resultados salvos em:", out_path)

    return {
        "key": key,
        "strategy": "dropna",
        "variant": "default",
        "rows_original": int(df.shape[0]),
        "rows_after_prepros": int(df_clean.shape[0]),
        "results": results,
    }

In [23]:

print(f"{"=" * 5} DROPNA {"=" * 5}")

dataset="datasets/sinistros/datatran2025"
target="teve_mortos"
out_dir = "data"
test_size = 0.2
random_state = 42

all_runs = []
try:
    run_info = run_one_dataset_with_drop(
        key=dataset,
        target=target,
        out_dir=out_dir,
        test_size=test_size,
        random_state=random_state,
    )
    all_runs.append(run_info)
except Exception as e:
    print(f"\n[ERRO] key={dataset}: {e}")
    all_runs.append({"key": dataset, "error": str(e)})
    


# --- Monta tabelas separadas por tipo de tarefa ---

classification_rows = []
regression_rows = []

for r in all_runs:
    base = {"experiment": "dropna", "key": r["key"], "variant": r.get("variant", "default"), "best_model": "", "error": r.get("error", "")}

    if "error" in r and "results" not in r:
        # Dataset falhou: adiciona na tabela correta com metricas vazias
        classification_rows.append({**base, "accuracy": None, "macro_f1": None, "weighted_f1": None, "weighted_precision": None, "weighted_recall": None})
        continue

    best = max(r["results"], key=lambda m: r["results"][m]["weighted_f1"])
    m = r["results"][best]
    classification_rows.append({
        "experiment": "dropna",
        "key": r["key"],
        "variant": r["variant"],
        "best_model": best,
        "accuracy": round(m["accuracy"], 4),
        "macro_f1": round(m["macro_f1"], 4),
        "weighted_f1": round(m["weighted_f1"], 4),
        "weighted_precision": round(m["weighted_precision"], 4),
        "weighted_recall": round(m["weighted_recall"], 4),
        "error": "",
    })

# Exibe  resultados e salva CSVs 

data_path = Path(out_dir)

if classification_rows:
    clf_df = pd.DataFrame(classification_rows, columns=["experiment", "key", "variant", "best_model", "accuracy", "macro_f1", "weighted_f1", "weighted_precision", "weighted_recall", "error"])
    print("\n=== Classificacao ===")
    display(clf_df)
    clf_path = data_path / "classification_summary.csv"
    clf_df.to_csv(clf_path, index=False)
    print("Salvo em:", clf_path)


print(f"{"=" * 5} FIM DROPNA {"=" * 5}")

===== DROPNA =====

Dataset: datasets/sinistros/datatran2025 | Target: teve_mortos
Shape original: (72529, 36)
Linhas removidas no dropna: 39 (0.05%)
Distribuicao da target:
teve_mortos
False    67281
True      5209
Name: count, dtype: int64

[model] knn
{'accuracy': 0.9759277141674714, 'macro_f1': 0.8939378164107048, 'weighted_f1': 0.9737858269820157, 'weighted_precision': 0.9762365593619559, 'weighted_recall': 0.9759277141674714}
Matriz de confusao:
[[13448     8]
 [  341   701]]
              precision    recall  f1-score   support

       False       0.98      1.00      0.99     13456
        True       0.99      0.67      0.80      1042

    accuracy                           0.98     14498
   macro avg       0.98      0.84      0.89     14498
weighted avg       0.98      0.98      0.97     14498


[model] random_forest
{'accuracy': 0.9909642709339219, 'macro_f1': 0.9640397850011453, 'weighted_f1': 0.9906831144845641, 'weighted_precision': 0.9910513895405058, 'weighted_recall': 0.

,experiment,key,variant,best_model,accuracy,macro_f1,weighted_f1,weighted_precision,weighted_recall,error
0,dropna,datasets/sinistros/datatran2025,default,,None,None,None,None,None,[Errno 2] Arquivo ou diretório inexistente: 'd...


Salvo em: data/classification_summary.csv
===== FIM DROPNA =====


# Integração com fontes externas

## Inmet

A primeira fonte de integração externa a ser utilizada será o INMET, que fornecerá dados acerca das condições climáticas no momento do acidente.

In [24]:
from pathlib import Path
import re

import pandas as pd

INMET_YEARS = [2025]  # edite esta lista para incluir novos anos
INMET_RAW_ROOT = Path("./data/datasets/inmet_raw")
INMET_PROCESSED_ROOT = Path("./data/datasets/inmet_processed")
INMET_FILENAME_RE = re.compile(
    r"^INMET_(?P<regiao>[^_]+)_(?P<uf>[^_]+)_(?P<estacao_id>[^_]+)_(?P<cidade>.+)_(?P<periodo>\d{2}-\d{2}-\d{4}_A_\d{2}-\d{2}-\d{4})\.CSV$",
    re.IGNORECASE,
)


def normalize_inmet_token(value: str) -> str:
    return re.sub(r"\s+", "-", value.strip())



def parse_inmet_filename(filename: str) -> dict[str, str]:
    match = INMET_FILENAME_RE.match(filename)
    if match is None:
        raise ValueError(f"Nome de arquivo INMET fora do padrao esperado: {filename}")

    info = match.groupdict()
    return {
        "regiao": info["regiao"].upper(),
        "uf": info["uf"].upper(),
        "estacao_id": info["estacao_id"].upper(),
        "cidade": normalize_inmet_token(info["cidade"]),
        "periodo": info["periodo"],
    }



def build_inmet_output_stem(filename: str) -> str:
    info = parse_inmet_filename(filename)
    return (
        f"inmet_{info['regiao']}_{info['uf']}_{info['estacao_id']}_"
        f"{info['cidade']}_{info['periodo']}_dados"
    )



def load_inmet_station_metadata(path: Path) -> pd.DataFrame:
    metadata = pd.read_csv(
        path,
        sep=";",
        encoding="latin1",
        nrows=8,
        header=None,
        names=["coluna", "valor"],
        engine="python",
    )
    metadata["coluna"] = metadata["coluna"].astype(str).str.replace(":", "", regex=False).str.strip()
    metadata["valor"] = metadata["valor"].astype(str).str.strip()
    return metadata



def load_inmet_data(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, sep=";", encoding="latin1", skiprows=8, engine="python")
    df = df.dropna(axis=1, how="all")
    df.columns = [str(column).strip() for column in df.columns]
    return df



def preprocess_inmet_file(path: Path, output_dir: Path) -> dict[str, str]:
    output_dir.mkdir(parents=True, exist_ok=True)

    output_stem = build_inmet_output_stem(path.name)
    metadata_path = output_dir / f"{output_stem}_station_metadata.csv"
    data_path = output_dir / f"{output_stem}.csv"

    metadata_df = load_inmet_station_metadata(path)
    data_df = load_inmet_data(path)

    metadata_df.to_csv(metadata_path, index=False)
    data_df.to_csv(data_path, index=False)

    return {
        "input_file": path.name,
        "metadata_path": str(metadata_path),
        "data_path": str(data_path),
    }



def preprocess_inmet_directory(year: int | str, source_root: Path, output_root: Path) -> list[dict[str, str]]:
    year_label = str(year)
    source_dir = source_root / year_label
    output_dir = output_root / year_label

    if not source_dir.exists():
        raise FileNotFoundError(f"Diretorio INMET nao encontrado: {source_dir}")

    csv_files = sorted(
        path for path in source_dir.iterdir()
        if path.is_file() and path.suffix.lower() == ".csv"
    )

    results: list[dict[str, str]] = []
    for path in csv_files:
        try:
            item = preprocess_inmet_file(path, output_dir)
            item["year"] = year_label
            results.append(item)
        except Exception as exc:
            print(f"[ERRO] {path.name}: {exc}")

    print(f"Processados {len(results)} de {len(csv_files)} arquivos INMET em {output_dir}")
    return results


inmet_results = []
for year in INMET_YEARS:
    try:
        inmet_results.extend(preprocess_inmet_directory(year, INMET_RAW_ROOT, INMET_PROCESSED_ROOT))
    except Exception as exc:
        print(f"[ERRO] ano={year}: {exc}")

inmet_results[:3]


Processados 594 de 594 arquivos INMET em data/datasets/inmet_processed/2025


[{'input_file': 'INMET_CO_DF_A001_BRASILIA_01-01-2025_A_31-12-2025.CSV',
  'metadata_path': 'data/datasets/inmet_processed/2025/inmet_CO_DF_A001_BRASILIA_01-01-2025_A_31-12-2025_dados_station_metadata.csv',
  'data_path': 'data/datasets/inmet_processed/2025/inmet_CO_DF_A001_BRASILIA_01-01-2025_A_31-12-2025_dados.csv',
  'year': '2025'},
 {'input_file': 'INMET_CO_DF_A042_BRAZLANDIA_01-01-2025_A_31-12-2025.CSV',
  'metadata_path': 'data/datasets/inmet_processed/2025/inmet_CO_DF_A042_BRAZLANDIA_01-01-2025_A_31-12-2025_dados_station_metadata.csv',
  'data_path': 'data/datasets/inmet_processed/2025/inmet_CO_DF_A042_BRAZLANDIA_01-01-2025_A_31-12-2025_dados.csv',
  'year': '2025'},
 {'input_file': 'INMET_CO_DF_A045_AGUAS EMENDADAS_01-01-2025_A_31-12-2025.CSV',
  'metadata_path': 'data/datasets/inmet_processed/2025/inmet_CO_DF_A045_AGUAS-EMENDADAS_01-01-2025_A_31-12-2025_dados_station_metadata.csv',
  'data_path': 'data/datasets/inmet_processed/2025/inmet_CO_DF_A045_AGUAS-EMENDADAS_01-01-2025_